# Ajuste de Hiperparámetros - Validación Estática

Utilizaremos `RandomizedSearchCV` junto con `PredefinedSplit` para asegurar que la búsqueda evalúe las combinaciones en nuestro conjunto de validación fijo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

### 1. Carga de Datos

### Desglose paso a paso:

1. **`pd.read_csv(base_path + 'y_train.csv')`**
Carga el archivo CSV como un `DataFrame` de Pandas con dimensiones **$(N, 1)$** (es decir, $N$ filas y 1 columna).
2. **`.values`**
Extrae la matriz subyacente en formato de arreglo de NumPy (`numpy.ndarray`). En este punto, la estructura sigue siendo bidimensional con dimensiones **$(N, 1)$** (una matriz columna).
3. **`.ravel()`**
Aplana la matriz bidimensional **$(N, 1)$** convirtiéndola en un vector unidimensional de dimensiones **$(N,)$**.

---

### ¿Por qué es necesario en Machine Learning?

La mayoría de los clasificadores y estimadores de Scikit-Learn (como `RandomForestClassifier`, `GradientBoostingClassifier` o `LogisticRegression`) esperan que la variable objetivo $y$ sea un arreglo unidimensional de forma `(N,)`.

Si le pasas un vector columna de forma `(N, 1)`, Scikit-Learn suele lanzar un aviso de advertencia (*DataConversionWarning*):

> *A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ).*

El uso de `.values.ravel()` elimina esa advertencia y asegura un formato limpio de NumPy para el entrenamiento.


In [ ]:
base_path = '../data/model_input/static/'

X_train = pd.read_csv(base_path + 'X_train.csv')
y_train = pd.read_csv(base_path + 'y_train.csv').values.ravel()

X_val = pd.read_csv(base_path + 'X_val.csv')
y_val = pd.read_csv(base_path + 'y_val.csv').values.ravel()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

### 2. Preprocesamiento y Configuración de Validación

Cuando usas `RandomizedSearchCV` (o `GridSearchCV`), por defecto Scikit-Learn intenta hacer una __validación cruzada (Cross-Validation)__ dividiendo los datos de entrenamiento en varios pedazos (folds).

Sin embargo, en tu enfoque estático, tú __ya tienes__ un conjunto de validación separado (`X_val` y `y_val`) y no quieres que Scikit-Learn haga validación cruzada. Quieres que entrene __siempre__ con `X_train` y evalúe __siempre__ con `X_val`.

### La solución: `PredefinedSplit`

Para obligar a `RandomizedSearchCV` a usar tu conjunto de validación fijo, usamos una herramienta llamada `PredefinedSplit`. Para que funcione, necesita que le pasemos todos los datos juntos (Train + Validation) y un arreglo especial que le diga a qué conjunto pertenece cada fila.

Aquí está el desglose de lo que hace el código:

1. __Concatenar los datos:__

   ```python
   X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
   y_train_val = np.concatenate([y_train, y_val])
   ```

   Juntamos `X_train` y `X_val` en un solo DataFrame gigante llamado `X_train_val`. Hacemos lo mismo con la variable objetivo `y`.

2. __Crear el arreglo de índices (`test_fold`):__

   ```python
   test_fold = np.concatenate([
       np.full(X_train.shape[0], -1), # -1 significa "Esto es para entrenar"
       np.full(X_val.shape[0], 0)     # 0 significa "Esto es para validar"
   ])
   ```

   Creamos una lista del mismo tamaño que `X_train_val`.

   - A todas las filas que venían de `X_train` les ponemos un __`-1`__. En el lenguaje de Scikit-Learn, `-1` significa *"usa esto solo para entrenar, no lo uses para evaluar"*.
   - A todas las filas que venían de `X_val` les ponemos un __`0`__. El `0` significa *"este es el fold 0 de validación"*.

3. __Configurar el Splitter:__

   ```python
   ps = PredefinedSplit(test_fold)
   ```

   Le pasamos esa lista a `PredefinedSplit`. Ahora, cuando le pasamos `cv=ps` a `RandomizedSearchCV`, el buscador sabe exactamente qué hacer: *"Ah, entreno con las filas marcadas con -1 y evalúo el AUC con las filas marcadas con 0"*.


In [ ]:
# Concatenar Train y Validation para PredefinedSplit
X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_val = np.concatenate([y_train, y_val])

test_fold = np.concatenate([
    np.full(X_train.shape[0], -1),
    np.full(X_val.shape[0], 0)
])
ps = PredefinedSplit(test_fold)

vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = [
    'is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

### 3. Random Forest - RandomizedSearchCV

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    'classifier__n_estimators': [100, 200, 300, 500],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

rf_random = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=ps,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

rf_random.fit(X_train_val, y_train_val)

print("Mejores hiperparámetros para Random Forest:")
print(rf_random.best_params_)
print(f"Mejor AUC: {rf_random.best_score_:.4f}")

### 4. Gradient Boosting - RandomizedSearchCV

In [ ]:
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

gb_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 5, 7],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

gb_random = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_param_grid,
    n_iter=15,
    cv=ps,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

gb_random.fit(X_train_val, y_train_val)

print("Mejores hiperparámetros para Gradient Boosting:")
print(gb_random.best_params_)
print(f"Mejor AUC: {gb_random.best_score_:.4f}")

### 5. Evaluación Detallada en Validación

In [ ]:
def evaluate_model(model, X, y, model_name):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    print(f"--- {model_name} ---")
    print(f"ROC AUC: {roc_auc_score(y, y_proba):.4f}")
    print(f"Accuracy: {accuracy_score(y, y_pred):.4f}\n")
    print(classification_report(y, y_pred))

# 3. Evaluar usando X_test_full en lugar de X_val
evaluate_model(rf_random.best_estimator_, X_val, y_val, "Random Forest (Tuned Test)")
evaluate_model(gb_random.best_estimator_, X_val, y_val, "Gradient Boosting (Tuned Test)")